1. Data load karo (churn-labeled customer summary):

In [2]:
import pandas as pd

df = pd.read_csv('../data/processed/customer_churn_labeled.csv')
print(df.head())
print(df.shape)

   Customer ID  Frequency  Monetary  Recency  Churned
0        12346         12  77556.46      325        1
1        12347          8   4921.53        1        0
2        12348          5   2019.40       74        0
3        12349          4   4428.69       18        0
4        12350          1    334.40      309        1
(5881, 5)


1. Mean — average value:

In [3]:
mean_monetary = df['Monetary'].mean()
print(f"Mean Monetary: {mean_monetary:.2f}")

Mean Monetary: 2954.40


Note: Mean sab values ka sum divide by count hota hai — lekin outliers se bahut affect hota hai. Agar ek customer ka spend 1,00,000 hai aur baaki sabka 500-1000, to mean bahut zyada bhaari lagega, jo "typical" customer ko represent nahi karega.

2. Median — middle value (outliers se resistant):

In [4]:
median_monetary = df['Monetary'].median()
print(f"Median Monetary: {median_monetary:.2f}")
print(f"Mean vs Median difference: {mean_monetary - median_monetary:.2f}")

Median Monetary: 865.60
Mean vs Median difference: 2088.80


Note: Agar Mean aur Median mein bada fark hai, iska matlab data skewed hai (kuch bahut high-value outliers hain jo mean ko upar khींच rahe hain). E-commerce spend data mein yeh bahut common hota hai — chunky "whale" customers.

3. Mode — sabse zyada frequent value:

In [6]:
mode_frequency = df['Frequency'].mode()
print(f"Mode Frequency: {mode_frequency.values}")

# Country jaisa categorical column mein bhi mode useful hai
churn_labels_full = pd.read_csv('../data/processed/step4_datatypes_fixed.csv', dtype={'Invoice': str, 'StockCode': str})
mode_country = churn_labels_full['Country'].mode()[0]
print(f"Most common country: {mode_country}")

Mode Frequency: [1]
Most common country: United Kingdom


4. Standard Deviation — data kitna "spread" hai average se:

In [7]:
std_monetary = df['Monetary'].std()
print(f"Standard Deviation: {std_monetary:.2f}")
print(f"Coefficient of Variation (std/mean): {(std_monetary/mean_monetary):.2f}")

Standard Deviation: 14437.32
Coefficient of Variation (std/mean): 4.89


Note: Chhota std deviation matlab values mean ke aas-paas clustered hain; bada std deviation matlab bahut variation hai (kuch bahut kam, kuch bahut zyada spend karne wale). "Coefficient of Variation" (std/mean) se different columns ka spread compare karna aasan hota hai (unit-independent measure).

5. Variance — std deviation ka squared version:

In [8]:
variance_monetary = df['Monetary'].var()
print(f"Variance: {variance_monetary:.2f}")
print(f"Std² should equal Variance: {std_monetary**2:.2f}")

Variance: 208436284.87
Std² should equal Variance: 208436284.87


6. Sab kuch ek saath, teeno RFM columns ke liye:

In [9]:
stats_summary = df[['Recency', 'Frequency', 'Monetary']].agg(['mean', 'median', 'std', 'min', 'max'])
print(stats_summary)

           Recency   Frequency       Monetary
mean    200.457745    6.287196    2954.396237
median   95.000000    3.000000     865.600000
std     209.474135   13.012879   14437.322635
min       0.000000    1.000000       0.000000
max     738.000000  398.000000  580987.040000


7. Churned vs Not-Churned groups ke stats compare karna (business-relevant insight):

In [10]:
comparison = df.groupby('Churned')[['Recency', 'Frequency', 'Monetary']].agg(['mean', 'median'])
print(comparison)

            Recency        Frequency            Monetary         
               mean median      mean median         mean   median
Churned                                                          
0         30.888735   24.0  9.628887    5.0  4834.093697  1659.08
1        364.747238  378.0  3.049548    2.0  1133.223002   481.88


Yeh dikhayega ki churned customers ka average Frequency/Monetary kaisa tha non-churned se compare mein — ek bahut important business insight jo aage README mein bhi likhoge.

8. Skewness check — quick visual-free tareeka (agar Mean >> Median, to right-skewed hai):

In [11]:
skew_value = df['Monetary'].skew()
print(f"Skewness of Monetary: {skew_value:.2f}")

Skewness of Monetary: 25.08


Samajhna: Skewness 0 ke close ho to symmetric distribution; positive skew (jaise yahan expect hai) matlab kuch bahut high spenders hain jo tail ko right side khींch rahe hain.

Practice questions:

1. Recency column ke liye mean, median, std nikaalo — kya iska distribution bhi skewed hai?

- Mean > Median: Recency distribution aam taur par right-skewed (positive skew) hota hai. Iska matlab hai ki zyadatar active customers ki recency kam hoti hai (left side clustered), jabki inactive/churned customers ki recency bahut zyada hoti hai jo tail ko right side khinch deti hai (jisse Mean, Median se kaafi bada ho jata hai).

In [14]:
mean_recency = df['Recency'].mean()
median_recency = df['Recency'].median()
std_recency = df['Recency'].std()
skew_recency = df['Recency'].skew()

print(f"Mean Recency: {mean_recency:.2f}")
print(f"Median Recency: {median_recency:.2f}")
print(f"Std Recency: {std_recency:.2f}")
print(f"Skewness: {skew_recency:.2f}")

Mean Recency: 200.46
Median Recency: 95.00
Std Recency: 209.47
Skewness: 0.89


2. Churned=1 vs Churned=0 groups ka Monetary ka standard deviation compare karo — kaunsa group zyada "spread out" hai?

- Kaunsa group zyada "spread out" hai? Aam taur par Non-Churned (Churned=0) group ka standard deviation aur spread zyada hota hai. Kyunki is group mein loyal "whale" customers hote hain jo lambe samay se bar-bar high amounts spend kar rahe hote hain, jisse monetary values mein bada spread (variation) dikhta hai.

In [15]:
monetary_std_comp = df.groupby('Churned')['Monetary'].agg(['mean', 'median', 'std'])
print(monetary_std_comp)

                mean   median           std
Churned                                    
0        4834.093697  1659.08  20165.728538
1        1133.223002   481.88   3117.246106
